# v1.1 Population Methodology

This notebook defines the population methodology for Seattle Public Safety
Dashboard v1.1.

## Goals

We need defensible population denominators for:

1. Seattle citywide crime rates.
2. Crime rates for each SPD MCPP neighborhood.
3. Neighborhood comparisons and rankings.
4. Future current-vs-previous period comparisons.

## Geography problem

The dashboard uses SPD Micro-Community Policing Plan (MCPP) neighborhoods.

MCPP neighborhoods are not Census geographies and should not be assumed to
match Seattle Community Reporting Areas or commonly used neighborhood
boundaries.

Therefore, Census population estimates must be translated onto the MCPP
geography.

## Candidate methodology

Neighborhood population:

2024 ACS 5-Year block-group population
        ↓
2020 Census block population weights
        ↓
estimated 2024 population by Census block
        ↓
assign Census blocks to MCPP neighborhoods
        ↓
aggregate into MCPP population estimates

Citywide population:

- 2024 ACS 5-Year Seattle population for consistency with neighborhood rates.
- 2026 Washington OFM official Seattle population as the current official
  citywide population estimate.

The notebook will determine whether the MCPP estimates should be calibrated
to the ACS Seattle city total before production use.

In [1]:
from pathlib import Path
import os
import sys

import geopandas as gpd
import numpy as np
import pandas as pd
import requests
from IPython.display import display


# -------------------------------------------------------------------
# Locate repository root.
# -------------------------------------------------------------------

cwd = Path.cwd().resolve()

REPO_ROOT = next(
    (
        path
        for path in [cwd, *cwd.parents]
        if (path / "dashboard").is_dir()
        and (path / "app.py").exists()
    ),
    None,
)

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate repository root. "
        "Expected app.py and dashboard/."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")


# -------------------------------------------------------------------
# Project imports.
# -------------------------------------------------------------------

from dashboard.crime_dashboard_data import (
    load_crime_dashboard_context,
    normalize_neighborhood_name,
)


# -------------------------------------------------------------------
# Source constants.
# -------------------------------------------------------------------

ACS_YEAR = 2024

STATE_FIPS = "53"
KING_COUNTY_FIPS = "033"
SEATTLE_PLACE_FIPS = "63000"

ACS_POPULATION_VARIABLE = "B01003_001E"
ACS_POPULATION_MOE_VARIABLE = "B01003_001M"

CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")

ACS_API_URL = (
    f"https://api.census.gov/data/"
    f"{ACS_YEAR}/acs/acs5"
)

DECENNIAL_PL_API_URL = (
    "https://api.census.gov/data/2020/dec/pl"
)

TIGER_2020_BLOCK_URL = (
    "https://www2.census.gov/geo/tiger/"
    "TIGER2020/TABBLOCK20/"
    "tl_2020_53_tabblock20.zip"
)


# -------------------------------------------------------------------
# Current official Seattle population.
#
# Washington OFM April 1, 2026 official estimate.
# -------------------------------------------------------------------

OFM_SEATTLE_POPULATION_2026 = 823_400
OFM_ESTIMATE_DATE = pd.Timestamp("2026-04-01")


# -------------------------------------------------------------------
# Cache large Census geography outside the repository.
# -------------------------------------------------------------------

CACHE_DIR = (
    Path.home()
    / ".cache"
    / "spd_dashboard_population"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

BLOCK_ZIP_PATH = (
    CACHE_DIR
    / "tl_2020_53_tabblock20.zip"
)


# -------------------------------------------------------------------
# Load current dashboard geography.
# -------------------------------------------------------------------

crime_context = (
    load_crime_dashboard_context()
)

mcpp = (
    crime_context["mcpp_boundaries"]
    .copy()
)

pd.set_option(
    "display.max_columns",
    50,
)

pd.set_option(
    "display.max_rows",
    100,
)

print()
print(
    "Census API key available:",
    bool(CENSUS_API_KEY),
)

print(
    "MCPP polygons:",
    len(mcpp),
)

Repository root: C:\Users\benca\code\PersonalPythonProjects\SPDCallDashboard

Census API key available: False
MCPP polygons: 58


In [2]:
print("MCPP GEOGRAPHY")
print("=" * 60)

print("CRS:", mcpp.crs)

print(
    "Polygon count:",
    len(mcpp),
)

print(
    "Invalid geometries:",
    (~mcpp.geometry.is_valid).sum(),
)

print(
    "Missing geometries:",
    mcpp.geometry.isna().sum(),
)

print()


mcpp_names = (
    mcpp[
        "mcpp_neighborhood"
    ]
    .astype("string")
    .sort_values()
    .reset_index(drop=True)
)

display(
    mcpp_names.to_frame(
        name="mcpp_neighborhood"
    )
)


duplicates = (
    mcpp[
        "mcpp_neighborhood"
    ]
    .value_counts()
    .loc[
        lambda s: s > 1
    ]
)

print()
print(
    "Duplicate MCPP names:",
    len(duplicates),
)

if not duplicates.empty:
    display(duplicates)

MCPP GEOGRAPHY
CRS: EPSG:4326
Polygon count: 58
Invalid geometries: 0
Missing geometries: 0



,mcpp_neighborhood
0,alaska junction
1,alki
2,ballard north
3,ballard south
4,belltown
5,bitterlake
6,brighton/dunlap
7,capitol hill
8,central area/squire park
9,chinatown/international district



Duplicate MCPP names: 0


## Geography decision

Population estimates must be generated for the same MCPP geography used by the
crime dashboard.

Seattle Community Reporting Areas are useful external references, but they are
not interchangeable with MCPP neighborhoods.

The population pipeline should therefore use the actual MCPP boundary polygons
loaded by the dashboard rather than attempting a neighborhood-name crosswalk.

In [3]:
if not CENSUS_API_KEY:
    raise RuntimeError(
        "CENSUS_API_KEY is not available. "
        "Set it in your local environment before "
        "running Census API cells."
    )


def census_get(
    base_url,
    *,
    get,
    for_clause,
    in_clause=None,
):
    params = {
        "get": get,
        "for": for_clause,
        "key": CENSUS_API_KEY,
    }

    if in_clause is not None:
        params["in"] = in_clause

    response = requests.get(
        base_url,
        params=params,
        timeout=60,
    )

    response.raise_for_status()

    payload = response.json()

    if len(payload) < 2:
        raise ValueError(
            "Census API returned no data rows."
        )

    return pd.DataFrame(
        payload[1:],
        columns=payload[0],
    )


print("Census API helper ready.")

RuntimeError: CENSUS_API_KEY is not available. Set it in your local environment before running Census API cells.

In [4]:
acs_bg = census_get(
    ACS_API_URL,
    get=(
        "NAME,"
        f"{ACS_POPULATION_VARIABLE},"
        f"{ACS_POPULATION_MOE_VARIABLE}"
    ),
    for_clause="block group:*",
    in_clause=(
        f"state:{STATE_FIPS} "
        f"county:{KING_COUNTY_FIPS} "
        "tract:*"
    ),
)


acs_bg["population"] = pd.to_numeric(
    acs_bg[
        ACS_POPULATION_VARIABLE
    ],
    errors="coerce",
)

acs_bg["population_moe"] = (
    pd.to_numeric(
        acs_bg[
            ACS_POPULATION_MOE_VARIABLE
        ],
        errors="coerce",
    )
)

acs_bg["bg_geoid"] = (
    acs_bg["state"]
    + acs_bg["county"]
    + acs_bg["tract"]
    + acs_bg["block group"]
)


print(
    "ACS block groups returned:",
    len(acs_bg),
)

print(
    "Missing population estimates:",
    acs_bg[
        "population"
    ].isna().sum(),
)

print(
    "King County ACS population sum:",
    f"{acs_bg['population'].sum():,.0f}",
)

display(
    acs_bg.head()
)


NameError: name 'census_get' is not defined

In [5]:
acs_seattle = census_get(
    ACS_API_URL,
    get=(
        "NAME,"
        f"{ACS_POPULATION_VARIABLE},"
        f"{ACS_POPULATION_MOE_VARIABLE}"
    ),
    for_clause=(
        f"place:{SEATTLE_PLACE_FIPS}"
    ),
    in_clause=(
        f"state:{STATE_FIPS}"
    ),
)


acs_seattle["population"] = (
    pd.to_numeric(
        acs_seattle[
            ACS_POPULATION_VARIABLE
        ],
        errors="coerce",
    )
)

acs_seattle[
    "population_moe"
] = pd.to_numeric(
    acs_seattle[
        ACS_POPULATION_MOE_VARIABLE
    ],
    errors="coerce",
)


ACS_SEATTLE_POPULATION = int(
    acs_seattle.iloc[0][
        "population"
    ]
)


print(
    "2024 ACS 5-Year Seattle population:",
    f"{ACS_SEATTLE_POPULATION:,}",
)

print(
    "2026 OFM official Seattle population:",
    f"{OFM_SEATTLE_POPULATION_2026:,}",
)

print(
    "Difference:",
    f"{OFM_SEATTLE_POPULATION_2026 - ACS_SEATTLE_POPULATION:,}",
)

print(
    "Difference (%):",
    (
        (
            OFM_SEATTLE_POPULATION_2026
            / ACS_SEATTLE_POPULATION
            - 1
        )
        * 100
    ),
)

NameError: name 'census_get' is not defined

## Why use 2020 Census blocks?

ACS block-group population cannot be assigned directly to an MCPP neighborhood
when a block group crosses an MCPP boundary.

A simple area-weighted split would implicitly assume people are distributed
uniformly throughout the block group. That is especially problematic in
Seattle because industrial areas, parks, water, and dense residential areas can
exist inside the same larger Census geography.

Instead:

1. Get the exact 2020 Census population of each Census block.
2. Calculate each block's share of its parent block group's 2020 population.
3. Apply that share to the block group's 2024 ACS population estimate.
4. Assign each small block to an MCPP neighborhood.

This is a population-weighted interpolation.

Important limitation:

The method assumes the within-block-group population distribution in 2024 is
approximately proportional to its 2020 distribution.

The notebook will identify block groups where that assumption cannot be used,
such as a 2020 zero-population block group with a positive 2024 ACS estimate.

In [6]:
blocks_population = census_get(
    DECENNIAL_PL_API_URL,
    get="NAME,P1_001N",
    for_clause="block:*",
    in_clause=(
        f"state:{STATE_FIPS} "
        f"county:{KING_COUNTY_FIPS} "
        "tract:*"
    ),
)


blocks_population[
    "population_2020"
] = pd.to_numeric(
    blocks_population[
        "P1_001N"
    ],
    errors="coerce",
)


blocks_population[
    "block_geoid"
] = (
    blocks_population["state"]
    + blocks_population["county"]
    + blocks_population["tract"]
    + blocks_population["block"]
)


# A Census block group's GEOID is:
#
# state + county + tract + first digit of block number

blocks_population[
    "bg_geoid"
] = (
    blocks_population[
        "block_geoid"
    ]
    .str[:12]
)


print(
    "King County Census blocks:",
    f"{len(blocks_population):,}",
)

print(
    "2020 King County population:",
    f"{blocks_population['population_2020'].sum():,.0f}",
)

print(
    "Zero-population blocks:",
    (
        blocks_population[
            "population_2020"
        ]
        == 0
    ).sum(),
)

display(
    blocks_population.head()
)

NameError: name 'census_get' is not defined

In [ ]:
def download_file(
    url,
    destination,
    chunk_size=1024 * 1024,
):
    destination = Path(destination)

    if destination.exists():
        print(
            "Using cached file:",
            destination,
        )

        return destination

    print(
        "Downloading:",
        url,
    )

    with requests.get(
        url,
        stream=True,
        timeout=120,
    ) as response:

        response.raise_for_status()

        with destination.open(
            "wb"
        ) as file:

            for chunk in response.iter_content(
                chunk_size=chunk_size
            ):
                if chunk:
                    file.write(chunk)

    print(
        "Saved:",
        destination,
    )

    return destination


download_file(
    TIGER_2020_BLOCK_URL,
    BLOCK_ZIP_PATH,
)

In [ ]:
# Convert the MCPP extent to the Census block CRS.
#
# TIGER 2020 blocks use NAD83 / EPSG:4269.

mcpp_4269 = (
    mcpp
    .to_crs(
        epsg=4269
    )
)

bbox = tuple(
    mcpp_4269.total_bounds
)


blocks_geo = gpd.read_file(
    f"zip://{BLOCK_ZIP_PATH}",
    bbox=bbox,
)


# Defensive county filter.

if "COUNTYFP20" in blocks_geo.columns:
    blocks_geo = (
        blocks_geo[
            blocks_geo[
                "COUNTYFP20"
            ]
            == KING_COUNTY_FIPS
        ]
        .copy()
    )


blocks_geo = (
    blocks_geo
    .rename(
        columns={
            "GEOID20": (
                "block_geoid"
            )
        }
    )
)


print(
    "Block geometries in Seattle extent:",
    f"{len(blocks_geo):,}",
)

print(
    "CRS:",
    blocks_geo.crs,
)

print(
    "Invalid geometries:",
    (~blocks_geo.geometry.is_valid).sum(),
)

In [ ]:
blocks = (
    blocks_geo[
        [
            "block_geoid",
            "geometry",
        ]
    ]
    .merge(
        blocks_population[
            [
                "block_geoid",
                "bg_geoid",
                "population_2020",
            ]
        ],
        on="block_geoid",
        how="left",
        validate="one_to_one",
    )
)


print(
    "Block geometries:",
    len(blocks),
)

print(
    "Missing 2020 population:",
    blocks[
        "population_2020"
    ].isna().sum(),
)


if blocks[
    "population_2020"
].isna().any():

    display(
        blocks.loc[
            blocks[
                "population_2020"
            ].isna(),
            [
                "block_geoid",
                "bg_geoid",
            ],
        ].head(50)
    )

In [ ]:
# Work in a projected CRS when creating representative points.

PROJECTED_CRS = "EPSG:2285"


blocks_projected = (
    blocks
    .to_crs(
        PROJECTED_CRS
    )
)


block_points = (
    blocks_projected[
        [
            "block_geoid",
            "bg_geoid",
            "population_2020",
            "geometry",
        ]
    ]
    .copy()
)


block_points[
    "geometry"
] = (
    block_points
    .geometry
    .representative_point()
)


mcpp_projected = (
    mcpp
    .to_crs(
        PROJECTED_CRS
    )
)


block_mcpp = (
    gpd.sjoin(
        block_points,
        mcpp_projected[
            [
                "mcpp_neighborhood",
                "geometry",
            ]
        ],
        how="inner",
        predicate="within",
    )
    .drop(
        columns=[
            "index_right",
        ],
        errors="ignore",
    )
)


print(
    "Blocks assigned to MCPP:",
    f"{len(block_mcpp):,}",
)

print(
    "2020 population represented:",
    f"{block_mcpp['population_2020'].sum():,.0f}",
)

print(
    "MCPP neighborhoods represented:",
    block_mcpp[
        "mcpp_neighborhood"
    ].nunique(),
)

In [ ]:
bg_population_2020 = (
    blocks_population
    .groupby(
        "bg_geoid",
        as_index=False,
    )
    .agg(
        bg_population_2020=(
            "population_2020",
            "sum",
        )
    )
)


block_mcpp = (
    block_mcpp
    .merge(
        bg_population_2020,
        on="bg_geoid",
        how="left",
        validate="many_to_one",
    )
)


block_mcpp[
    "population_weight"
] = np.where(
    block_mcpp[
        "bg_population_2020"
    ]
    > 0,

    block_mcpp[
        "population_2020"
    ]
    / block_mcpp[
        "bg_population_2020"
    ],

    np.nan,
)


display(
    block_mcpp[
        [
            "block_geoid",
            "bg_geoid",
            "mcpp_neighborhood",
            "population_2020",
            "bg_population_2020",
            "population_weight",
        ]
    ].head()
)

In [ ]:
block_mcpp = (
    block_mcpp
    .merge(
        acs_bg[
            [
                "bg_geoid",
                "population",
                "population_moe",
            ]
        ].rename(
            columns={
                "population": (
                    "acs_bg_population_2024"
                ),
                "population_moe": (
                    "acs_bg_population_moe_2024"
                ),
            }
        ),
        on="bg_geoid",
        how="left",
        validate="many_to_one",
    )
)


missing_acs_bg = (
    block_mcpp[
        "acs_bg_population_2024"
    ].isna()
)


print(
    "Seattle-area blocks with no matching "
    "2024 ACS block group:",
    missing_acs_bg.sum(),
)

print(
    "2020 population in unmatched blocks:",
    block_mcpp.loc[
        missing_acs_bg,
        "population_2020",
    ].sum(),
)


if missing_acs_bg.any():
    display(
        block_mcpp.loc[
            missing_acs_bg,
            [
                "block_geoid",
                "bg_geoid",
                "mcpp_neighborhood",
                "population_2020",
            ],
        ]
        .head(100)
    )

In [ ]:
problem_bg = (
    block_mcpp[
        (
            block_mcpp[
                "bg_population_2020"
            ]
            <= 0
        )
        & (
            block_mcpp[
                "acs_bg_population_2024"
            ]
            > 0
        )
    ]
    [
        [
            "bg_geoid",
            "acs_bg_population_2024",
            "mcpp_neighborhood",
        ]
    ]
    .drop_duplicates()
)


print(
    "ACS-positive block groups with zero "
    "2020 population denominator:",
    len(problem_bg),
)


if not problem_bg.empty:
    display(problem_bg)

In [ ]:
block_mcpp[
    "estimated_population_2024"
] = (
    block_mcpp[
        "acs_bg_population_2024"
    ]
    * block_mcpp[
        "population_weight"
    ]
)


print(
    "Estimated 2024 population assigned "
    "to MCPP blocks:",
    f"{block_mcpp['estimated_population_2024'].sum():,.1f}",
)

In [ ]:
mcpp_population_raw = (
    block_mcpp
    .groupby(
        "mcpp_neighborhood",
        as_index=False,
    )
    .agg(
        population_2024_raw=(
            "estimated_population_2024",
            "sum",
        ),
        census_blocks=(
            "block_geoid",
            "nunique",
        ),
        source_block_groups=(
            "bg_geoid",
            "nunique",
        ),
    )
)


mcpp_population_raw[
    "population_2024_raw"
] = (
    mcpp_population_raw[
        "population_2024_raw"
    ]
    .round()
    .astype(int)
)


mcpp_population_raw = (
    mcpp_population_raw
    .sort_values(
        "population_2024_raw",
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    mcpp_population_raw
)

In [ ]:
raw_mcpp_total = (
    mcpp_population_raw[
        "population_2024_raw"
    ].sum()
)


difference = (
    raw_mcpp_total
    - ACS_SEATTLE_POPULATION
)


pct_difference = (
    difference
    / ACS_SEATTLE_POPULATION
    * 100
)


reconciliation = pd.DataFrame(
    {
        "population_measure": [
            (
                "Raw reconstructed "
                "MCPP population"
            ),
            (
                "2024 ACS Seattle city"
            ),
            (
                "2026 OFM Seattle city"
            ),
        ],
        "population": [
            raw_mcpp_total,
            ACS_SEATTLE_POPULATION,
            OFM_SEATTLE_POPULATION_2026,
        ],
    }
)


display(reconciliation)


print()
print(
    "Raw MCPP minus ACS city:",
    f"{difference:+,.0f}",
)

print(
    "Difference from ACS city:",
    f"{pct_difference:+.3f}%",
)

In [ ]:
OLD_POPULATION_PATH = (
    REPO_ROOT
    / "data"
    / "external"
    / "neighborhood_population.csv"
)


old_population = pd.read_csv(
    OLD_POPULATION_PATH
)


old_population[
    "mcpp_neighborhood"
] = normalize_neighborhood_name(
    old_population[
        "dispatch_neighborhood"
    ]
)


old_population[
    "population"
] = pd.to_numeric(
    old_population[
        "population"
    ],
    errors="coerce",
)


old_population = (
    old_population[
        [
            "mcpp_neighborhood",
            "population",
        ]
    ]
    .rename(
        columns={
            "population": (
                "old_population"
            )
        }
    )
)


population_comparison = (
    mcpp_population_raw
    .merge(
        old_population,
        on="mcpp_neighborhood",
        how="outer",
    )
)


population_comparison[
    "raw_difference"
] = (
    population_comparison[
        "population_2024_raw"
    ]
    - population_comparison[
        "old_population"
    ]
)


population_comparison[
    "pct_difference"
] = (
    population_comparison[
        "raw_difference"
    ]
    / population_comparison[
        "old_population"
    ]
    * 100
)


display(
    population_comparison
    .sort_values(
        "pct_difference",
        key=lambda s: s.abs(),
        ascending=False,
        na_position="last",
    )
)


print()
print(
    "Old static population total:",
    f"{old_population['old_population'].sum():,.0f}",
)

print(
    "New raw MCPP estimate total:",
    f"{raw_mcpp_total:,.0f}",
)

In [ ]:
ACS_CALIBRATION_FACTOR = (
    ACS_SEATTLE_POPULATION
    / raw_mcpp_total
)


mcpp_population = (
    mcpp_population_raw
    .copy()
)


mcpp_population[
    "population_2024_acs_calibrated"
] = (
    mcpp_population[
        "population_2024_raw"
    ]
    * ACS_CALIBRATION_FACTOR
)


mcpp_population[
    "population_2024_acs_calibrated"
] = (
    mcpp_population[
        "population_2024_acs_calibrated"
    ]
    .round()
    .astype(int)
)


print(
    "ACS city calibration factor:",
    ACS_CALIBRATION_FACTOR,
)

print(
    "Calibration adjustment:",
    f"{(ACS_CALIBRATION_FACTOR - 1) * 100:+.3f}%",
)

print(
    "Calibrated neighborhood sum:",
    f"{mcpp_population['population_2024_acs_calibrated'].sum():,}",
)

print(
    "ACS Seattle city population:",
    f"{ACS_SEATTLE_POPULATION:,}",
)


display(
    mcpp_population[
        [
            "mcpp_neighborhood",
            "population_2024_raw",
            "population_2024_acs_calibrated",
        ]
    ]
)

In [ ]:
OFM_SCALE_FACTOR = (
    OFM_SEATTLE_POPULATION_2026
    / ACS_SEATTLE_POPULATION
)


mcpp_population[
    "population_2026_uniform_ofm_scaled"
] = (
    mcpp_population[
        "population_2024_acs_calibrated"
    ]
    * OFM_SCALE_FACTOR
)


mcpp_population[
    "population_2026_uniform_ofm_scaled"
] = (
    mcpp_population[
        "population_2026_uniform_ofm_scaled"
    ]
    .round()
    .astype(int)
)


print(
    "Uniform 2024 ACS -> 2026 OFM "
    "growth factor:",
    OFM_SCALE_FACTOR,
)

print(
    "Uniform implied growth:",
    f"{(OFM_SCALE_FACTOR - 1) * 100:.2f}%",
)


display(
    mcpp_population[
        [
            "mcpp_neighborhood",
            "population_2024_acs_calibrated",
            "population_2026_uniform_ofm_scaled",
        ]
    ]
)

## Important interpretation of OFM scaling

Scaling every MCPP neighborhood by the Seattle-wide 2024-to-2026 growth factor
would force the neighborhood estimates to sum to the current OFM city total.

However, it assumes every neighborhood grew at the same rate.

That assumption is unlikely to be literally true.

Therefore, the uniformly OFM-scaled neighborhood values should be treated as a
sensitivity scenario, not automatically adopted as the production denominator.

A cleaner default may be:

- Neighborhood crime rates:
  2024 ACS-based MCPP estimates.

- Citywide crime rate:
  2024 ACS Seattle population when direct comparability with neighborhood
  rates is important.

- Current Seattle population context:
  2026 OFM official city population.

The final decision should depend on the reconciliation diagnostics above.

In [ ]:
all_mcpp = set(
    normalize_neighborhood_name(
        mcpp[
            "mcpp_neighborhood"
        ]
    )
    .dropna()
)


estimated_mcpp = set(
    normalize_neighborhood_name(
        mcpp_population[
            "mcpp_neighborhood"
        ]
    )
    .dropna()
)


missing_population_neighborhoods = sorted(
    all_mcpp
    - estimated_mcpp
)


unexpected_population_neighborhoods = sorted(
    estimated_mcpp
    - all_mcpp
)


print(
    "MCPP polygons:",
    len(all_mcpp),
)

print(
    "MCPPs with population estimates:",
    len(estimated_mcpp),
)

print()
print(
    "Missing population neighborhoods:"
)

print(
    missing_population_neighborhoods
)

print()
print(
    "Unexpected population neighborhoods:"
)

print(
    unexpected_population_neighborhoods
)

In [ ]:
small_population = (
    mcpp_population[
        mcpp_population[
            "population_2024_acs_calibrated"
        ]
        < 500
    ]
    .copy()
)


display(
    small_population[
        [
            "mcpp_neighborhood",
            "population_2024_raw",
            "population_2024_acs_calibrated",
            "census_blocks",
            "source_block_groups",
        ]
    ]
    .sort_values(
        "population_2024_acs_calibrated"
    )
)

In [ ]:
relevant_bg = (
    block_mcpp[
        [
            "bg_geoid",
            "acs_bg_population_2024",
            "acs_bg_population_moe_2024",
        ]
    ]
    .drop_duplicates()
    .copy()
)


relevant_bg[
    "relative_moe_pct"
] = (
    relevant_bg[
        "acs_bg_population_moe_2024"
    ]
    / relevant_bg[
        "acs_bg_population_2024"
    ]
    .replace(
        0,
        np.nan,
    )
    * 100
)


display(
    relevant_bg[
        "relative_moe_pct"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
        ]
    )
)


print(
    "Block groups with >25% relative MOE:",
    (
        relevant_bg[
            "relative_moe_pct"
        ]
        > 25
    ).sum(),
)

In [ ]:
candidate_population = (
    mcpp_population[
        [
            "mcpp_neighborhood",
            "population_2024_acs_calibrated",
        ]
    ]
    .rename(
        columns={
            "population_2024_acs_calibrated": (
                "population"
            )
        }
    )
    .copy()
)


candidate_population[
    "population_year"
] = ACS_YEAR

candidate_population[
    "geography_type"
] = "mcpp"

candidate_population[
    "source"
] = (
    "U.S. Census Bureau ACS 5-Year"
)

candidate_population[
    "source_vintage"
] = ACS_YEAR

candidate_population[
    "estimation_method"
] = (
    "ACS block-group population distributed using "
    "2020 Census block population weights and "
    "calibrated to ACS Seattle city population"
)


city_row = pd.DataFrame(
    {
        "mcpp_neighborhood": [
            pd.NA
        ],
        "population": [
            ACS_SEATTLE_POPULATION
        ],
        "population_year": [
            ACS_YEAR
        ],
        "geography_type": [
            "city"
        ],
        "source": [
            "U.S. Census Bureau ACS 5-Year"
        ],
        "source_vintage": [
            ACS_YEAR
        ],
        "estimation_method": [
            "direct Seattle place estimate"
        ],
    }
)


candidate_population = pd.concat(
    [
        candidate_population,
        city_row,
    ],
    ignore_index=True,
)


display(
    candidate_population
)

In [ ]:
official_city_population = pd.DataFrame(
    {
        "geography_type": [
            "city"
        ],
        "geography_name": [
            "Seattle"
        ],
        "population": [
            OFM_SEATTLE_POPULATION_2026
        ],
        "estimate_date": [
            OFM_ESTIMATE_DATE
        ],
        "source": [
            (
                "Washington State Office "
                "of Financial Management"
            )
        ],
        "estimate_type": [
            "Official April 1 population estimate"
        ],
    }
)


display(
    official_city_population
)

In [ ]:
population_methodology_status = pd.DataFrame(
    [
        {
            "question": (
                "Use current dashboard MCPP boundaries?"
            ),
            "candidate_decision": "Yes",
            "status": "review",
        },
        {
            "question": (
                "Neighborhood population source?"
            ),
            "candidate_decision": (
                "2024 ACS 5-Year block groups"
            ),
            "status": "review",
        },
        {
            "question": (
                "Spatial interpolation?"
            ),
            "candidate_decision": (
                "2020 Census block population weights"
            ),
            "status": "review",
        },
        {
            "question": (
                "Calibrate MCPP estimates to ACS city total?"
            ),
            "candidate_decision": (
                "Likely yes if reconciliation adjustment "
                "is small"
            ),
            "status": "review",
        },
        {
            "question": (
                "Use uniform OFM scaling for neighborhoods?"
            ),
            "candidate_decision": "Probably no",
            "status": "review",
        },
        {
            "question": (
                "Neighborhood rate denominator?"
            ),
            "candidate_decision": (
                "ACS-calibrated MCPP estimate"
            ),
            "status": "review",
        },
        {
            "question": (
                "City rate denominator?"
            ),
            "candidate_decision": (
                "2024 ACS for methodological "
                "consistency"
            ),
            "status": "review",
        },
        {
            "question": (
                "Current city population context?"
            ),
            "candidate_decision": (
                "2026 OFM official estimate"
            ),
            "status": "review",
        },
    ]
)


display(
    population_methodology_status
)

In [ ]:
print(
    "v1.1 POPULATION METHODOLOGY AUDIT"
)

print("=" * 60)

print()

print(
    "ACS vintage:",
    ACS_YEAR,
)

print(
    "ACS Seattle population:",
    f"{ACS_SEATTLE_POPULATION:,}",
)

print(
    "OFM Seattle population:",
    f"{OFM_SEATTLE_POPULATION_2026:,}",
)

print()

print(
    "Raw reconstructed MCPP total:",
    f"{raw_mcpp_total:,}",
)

print(
    "Raw reconstruction difference from ACS:",
    f"{pct_difference:+.3f}%",
)

print(
    "ACS calibration factor:",
    f"{ACS_CALIBRATION_FACTOR:.6f}",
)

print()

print(
    "MCPP neighborhoods:",
    len(all_mcpp),
)

print(
    "MCPP neighborhoods with estimates:",
    len(estimated_mcpp),
)

print(
    "Missing MCPP populations:",
    len(
        missing_population_neighborhoods
    ),
)

print()

print(
    "Unmatched Census blocks:",
    int(
        missing_acs_bg.sum()
    ),
)

print(
    "Zero-weight problem block groups:",
    len(problem_bg),
)

print()

print(
    "Existing static population total:",
    f"{old_population['old_population'].sum():,.0f}",
)

print()

print(
    "Population methodology is ready to finalize "
    "only if geography matching and weighting "
    "diagnostics above are acceptable."
)